# EDA trước tiền xử lý — Fraud Detection (3 datasets)

Notebook khảo sát **MLG-ULB**, **IEEE-CIS** và **Sparkov/Kartik2112** theo nền tảng trong `.agent`: cấu trúc, dung lượng, số dòng/bảng, kiểu feature, missing, duplicate, class imbalance, ID/time, leakage và ước lượng One-Hot + SMOTE trước TabNet.

**Ranh giới:** chỉ EDA. Không drop/impute/encode/scale, không tạo mẫu SMOTE, không split chính thức và không train model. Các đề xuất là bằng chứng để khóa preprocessing ở phase sau.

## Cách hiểu ước lượng RAM

- `Dense float32 = rows × features × 4 bytes`; chưa gồm DataFrame, target, optimizer, batch hay bản sao tạm.
- `CSR upper` giả định mỗi numeric và mỗi nhóm One-Hot có một non-zero/dòng. Đây là storage estimate, **không phải peak RAM**.
- SMOTE được mô phỏng với `sampling_strategy=1.0` trên training portion 80%; không chạy SMOTE thật. Peak RAM nearest-neighbor có thể lớn hơn nhiều.
- SMOTE sau One-Hot có thể sinh dummy dạng phân số. Notebook ghi nhận rủi ro nhưng không tự đổi sang SMOTENC/encoder khác.
- One-Hot ID/high-cardinality là rủi ro lớn nhất trước TabNet, kể cả khi giữ sparse.

In [ ]:
from pathlib import Path
import gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
RANDOM_STATE = 42
TRAIN_FRAC = .80              # training portion của Stratified 5-Fold
EDA_NROWS = None             # None=FULL; 200_000 chỉ dùng smoke test
PLOT_SAMPLE = 200_000
LOCAL_INPUTS = [Path('data/Raw_data'), Path('../data/Raw_data')]
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else next((p for p in LOCAL_INPUTS if p.exists()), LOCAL_INPUTS[0])
print('Input:', INPUT_ROOT.resolve())
print('Mode:', 'FULL' if EDA_NROWS is None else f'SAMPLE {EDA_NROWS:,}')

In [ ]:
def bfmt(n):
    n=float(n)
    for u in ['B','KiB','MiB','GiB','TiB']:
        if n<1024 or u=='TiB': return f'{n:,.2f} {u}'
        n/=1024

def locate(name, hints=()):
    found=list(INPUT_ROOT.rglob(name))
    hinted=[p for p in found if any(h.lower() in str(p).lower() for h in hints)]
    found=hinted or found
    if not found: raise FileNotFoundError(f'Không thấy {name} dưới {INPUT_ROOT}. Hãy Add Input như ảnh.')
    if len(found)>1: print(f'[WARN] {len(found)} matches; dùng {found[0]}')
    return found[0]

def load(name, hints, label):
    p=locate(name,hints); print(f'\n{label}: {p} | file={bfmt(p.stat().st_size)}')
    d=pd.read_csv(p,nrows=EDA_NROWS,low_memory=False)
    print('shape=',d.shape,'| DataFrame RAM=',bfmt(d.memory_usage(deep=True).sum()))
    return d

def profile(d,label,target=None):
    num=d.select_dtypes(include=np.number).shape[1]
    out={'table':label,'rows':len(d),'columns':d.shape[1],'numeric':num,'non_numeric':d.shape[1]-num,
         'ram':bfmt(d.memory_usage(deep=True).sum()),'missing_cells':int(d.isna().sum().sum()),
         'duplicate_full_rows':int(d.duplicated().sum())}
    display(pd.DataFrame([out])); display(d.dtypes.astype(str).value_counts().rename('columns').to_frame())
    if target:
        assert target in d
        c=d[target].value_counts(dropna=False).sort_index(); display(pd.DataFrame({'count':c,'ratio':c/len(d)}))
        c.plot.bar(logy=True,color=['#4C78A8','#E45756'],title=f'{label}: class count (log)')
        plt.ylabel('count'); plt.show()
    return out

def missing(d,label,top=25):
    m=pd.DataFrame({'feature':d.columns,'dtype':d.dtypes.astype(str).values,
      'missing_count':d.isna().sum().values,'missing_ratio':d.isna().mean().values,
      'nunique':d.nunique(dropna=True).values}).sort_values('missing_ratio',ascending=False)
    display(m.head(top))
    bands=pd.cut(m.missing_ratio,[-.001,0,.1,.25,.5,.75,1],labels=['0%','(0,10%]','(10,25%]','(25,50%]','(50,75%]','(75,100%]'])
    display(bands.value_counts(sort=False).rename('columns').to_frame())
    m.head(top).sort_values('missing_ratio').plot.barh(x='feature',y='missing_ratio',figsize=(9,max(4,top*.2)),legend=False,title=label+' missing')
    plt.axvline(.5,color='red',ls='--'); plt.tight_layout(); plt.show(); return m

def attention(d,target,ids=(),times=()):
    n=d.nunique(dropna=True); r=n/max(len(d),1)
    a=pd.DataFrame({'feature':d.columns,'dtype':d.dtypes.astype(str).values,'nunique':n.values,
                    'unique_ratio':r.values,'missing_ratio':d.isna().mean().values}); a['flag']=''
    a.loc[a.feature.isin(ids),'flag']+='ID/PII; '; a.loc[a.feature.isin(times),'flag']+='time; '
    a.loc[(a.unique_ratio>.95)&(a.feature!=target),'flag']+='near-unique; '
    a.loc[(a.nunique<=1)&(a.feature!=target),'flag']+='constant; '
    a.loc[(a.missing_ratio>.5)&(a.feature!=target),'flag']+='missing>50%; '
    return a[a.flag!=''].sort_values(['flag','unique_ratio'],ascending=[True,False])

def amount_plot(d,col,target,label):
    s=d[[col,target]].dropna(); s=s.sample(min(len(s),PLOT_SAMPLE),random_state=42)
    s=s[s[col]<=s[col].quantile(.995)]
    for y,c in [(0,'#4C78A8'),(1,'#E45756')]: plt.hist(np.log1p(s.loc[s[target]==y,col].clip(lower=0)),60,density=True,alpha=.45,label=str(y),color=c)
    plt.title(label+': log1p('+col+') by class'); plt.legend(); plt.show()

def rate_table(d,col,target,min_n=100):
    z=d.groupby(col,dropna=False)[target].agg(['size','sum','mean']).rename(columns={'size':'count','sum':'fraud','mean':'fraud_rate'})
    return z[z['count']>=min_n].sort_values(['fraud_rate','count'],ascending=False).head(20)

def cats(d,exclude=()): return [c for c in d.select_dtypes(include=['object','category','bool']) if c not in exclude]

def estimate(parts,nrows,target,exclude,specs,scenario):
    # specs=(DataFrame,column,identity_join_adds_Unknown)
    total_cols=sum(c not in {target,*exclude} for d in parts for c in d.columns)
    levels=[]
    for d,c,join_unknown in specs:
        levels.append((c,int(d[c].nunique(dropna=True))+int(d[c].isna().any() or join_unknown)))
    n_num=total_cols-len(specs); n_ohe=sum(v for _,v in levels); nf=n_num+n_ohe; nr=round(nrows*TRAIN_FRAC)
    dense=nr*nf*4; csr=nr*(n_num+len(specs))*8+(nr+1)*4
    return {'scenario':scenario,'train_rows':nr,'numeric_features':n_num,'categorical_columns':len(specs),
      'ohe_features':n_ohe,'total_features':nf,'dense_float32':bfmt(dense),'csr_upper':bfmt(csr),
      '_dense':dense,'_csr':csr,'_levels':levels}

def smote_size(y,e):
    c=y.value_counts(); mi=int(c.min()*TRAIN_FRAC); ma=int(c.max()*TRAIN_FRAC); before=mi+ma; after=2*ma; mult=after/before
    return {'train_majority':ma,'train_minority':mi,'synthetic_rows':ma-mi,'rows_after':after,'row_multiplier':mult,
            'dense_after':bfmt(e['_dense']*mult),'csr_after_rough':bfmt(e['_csr']*mult)}

def public(e): return {k:v for k,v in e.items() if not k.startswith('_')}
results={}

## 1) MLG-ULB — `creditcard.csv`

Kỳ vọng từ source: 284.807 dòng, 31 cột, 492 fraud (~0,172%); `V1..V28` đã PCA/anonymized, còn `Time` và `Amount` ở thang gốc. Cần chú ý missing, duplicate, outlier và mất cân bằng.

In [ ]:
mlg=load('creditcard.csv',('creditcardfraud','credit-card-fraud'),'MLG-ULB')
p=profile(mlg,'MLG-ULB','Class'); display(mlg.head()); mm=missing(mlg,'MLG-ULB',15)
display(attention(mlg,'Class',times=('Time',)))
display(mlg.groupby('Class')[['Time','Amount']].agg(['count','mean','median','std','min','max']))
amount_plot(mlg,'Amount','Class','MLG-ULB')
corr=mlg.corr(numeric_only=True)['Class'].drop('Class').sort_values(); display(pd.concat([corr.head(8),corr.tail(8)]).rename('corr_Class').to_frame())
e=estimate([mlg],len(mlg),'Class',set(),[],'Không categorical → không OHE'); s=smote_size(mlg.Class,e)
display(pd.DataFrame([public(e)])); display(pd.DataFrame([s]))
results['MLG-ULB']={**p,'fraud':int(mlg.Class.sum()),'fraud_rate':float(mlg.Class.mean()),'features_after_review':e['total_features'],'dense_train':e['dense_float32'],'rows_after_smote':s['rows_after'],'dense_after_smote':s['dense_after']}
del mlg,mm; gc.collect()

### Nhận định MLG-ULB

- Không cần One-Hot; trọng tâm là scaling `Time`/`Amount`, duplicate và outlier.
- Nếu xóa duplicate, phải khóa quy tắc trước split để cùng mẫu không lọt qua hai fold.
- SMOTE 1:1 gần nhân đôi training majority nhưng kích thước vẫn nhỏ hơn hai dataset còn lại.
- `Time` không phải ID. Random stratified và temporal split trả lời hai câu hỏi khác nhau; phase sau phải khóa protocol.

## 2) IEEE-CIS — transaction và identity

`train_transaction.csv` là bảng chính có target; `train_identity.csv` bổ sung theo `TransactionID`. Notebook audit khóa/độ phủ nhưng không materialize full left join để tránh thêm một bản sao RAM lớn. Missing identity sau join được tính theo tổng transaction.

In [ ]:
tx=load('train_transaction.csv',('ieee','fraud-detection'),'IEEE transaction')
ident=load('train_identity.csv',('ieee','fraud-detection'),'IEEE identity')
pt=profile(tx,'IEEE transaction','isFraud'); pi=profile(ident,'IEEE identity'); display(tx.head(3)); display(ident.head(3))
assert tx.TransactionID.is_unique and ident.TransactionID.is_unique
txids=pd.Index(tx.TransactionID); iids=pd.Index(ident.TransactionID); matched=iids.isin(txids)
display(pd.DataFrame({'check':['tx rows','tx unique ID','identity rows','identity unique ID','identity IDs matched','tx without identity','expected left-join rows'],
 'value':[len(tx),tx.TransactionID.nunique(),len(ident),ident.TransactionID.nunique(),int(matched.sum()),int((~txids.isin(iids)).sum()),len(tx)]}))
print('Column collisions ngoài key:',sorted((set(tx)&set(ident))-{'TransactionID'}))
print('Expected merged columns:',tx.shape[1]+ident.shape[1]-1)

In [ ]:
mt=missing(tx,'IEEE transaction',30); mid=ident.loc[matched]
eff=pd.DataFrame({'feature':[c for c in ident if c!='TransactionID'],
 'missing_after_join':[len(tx)-mid[c].notna().sum() for c in ident if c!='TransactionID'],
 'missing_ratio_after_join':[1-mid[c].notna().sum()/len(tx) for c in ident if c!='TransactionID'],
 'nunique':[mid[c].nunique(dropna=True) for c in ident if c!='TransactionID']}).sort_values('missing_ratio_after_join',ascending=False)
display(eff.head(40)); print('tx cols missing>50%:',int((mt.missing_ratio>.5).sum()),'| identity effective:',int((eff.missing_ratio_after_join>.5).sum()))
display(attention(tx,'isFraud',('TransactionID',),('TransactionDT',)).head(80)); display(attention(ident,'__none__',('TransactionID',)).head(80))
display(tx.groupby('isFraud')[['TransactionAmt','TransactionDT']].agg(['count','mean','median','std','min','max'])); amount_plot(tx,'TransactionAmt','isFraud','IEEE')
for c in ['ProductCD','card4','card6','P_emaildomain','R_emaildomain']:
    if c in tx: display(Markdown('**`'+c+'`**')); display(rate_table(tx,c,'isFraud'))

### IEEE: capacity One-Hot + SMOTE

A = categorical theo dtype. B = thêm `card*`/`addr*` numeric-coded để review. `TransactionID` luôn loại khỏi estimate. Mỗi categorical identity thêm mức `Unknown` do left join. Chưa giả lập drop missing >50%, nên đây là upper scenario; ở preprocessing, danh sách drop phải học từ training fold.

In [ ]:
to=cats(tx,('isFraud','TransactionID')); io=cats(ident,('TransactionID',))
strict=[(tx,c,False) for c in to]+[(ident,c,True) for c in io]
coded=[c for c in tx if (c.startswith('card') or c.startswith('addr')) and c not in to and c not in ('isFraud','TransactionID')]
review=strict+[(tx,c,False) for c in coded]
ea=estimate([tx,ident],len(tx),'isFraud',{'TransactionID'},strict,'A dtype categorical')
eb=estimate([tx,ident],len(tx),'isFraud',{'TransactionID'},review,'B + card*/addr* numeric-coded')
display(pd.DataFrame([public(ea),public(eb)])); display(pd.DataFrame(eb['_levels'],columns=['feature','levels']).sort_values('levels',ascending=False).head(30))
si=smote_size(tx.isFraud,eb); display(pd.DataFrame([si]))
results['IEEE merged estimate']={**pt,'columns':tx.shape[1]+ident.shape[1]-1,'fraud':int(tx.isFraud.sum()),'fraud_rate':float(tx.isFraud.mean()),'features_after_review':eb['total_features'],'dense_train':eb['dense_float32'],'rows_after_smote':si['rows_after'],'dense_after_smote':si['dense_after']}
del tx,ident,txids,iids,mid,mt,eff; gc.collect()

### Nhận định IEEE-CIS

- Transaction quyết định số dòng/target; identity chỉ phủ một phần nên left join làm missing tăng mạnh.
- Drop >50% phải xác định từ train fold; numeric `-999`, categorical `Unknown` theo source.
- `TransactionID` chỉ truy vết. `TransactionDT` có temporal drift. `card*`/`addr*` lưu dạng số nhưng có tính category; dtype-only dễ bỏ sót.
- Dense hóa full OHE là rủi ro RAM. Sparse giảm storage nhưng không loại bỏ peak RAM encoder/SMOTE/batch TabNet.

## 3) Sparkov/Kartik2112 — `fraudTrain` và `fraudTest`

Hai bảng là split sẵn có. `Unnamed: 0`, `trans_num`, `cc_num`, tên/địa chỉ là ID/PII hoặc proxy có nguy cơ bùng nổ One-Hot và memorization. Notebook chỉ đánh dấu, không drop.

In [ ]:
st=load('fraudTrain.csv',('fraud-detection','credit-card-transactions'),'Sparkov train')
sv=load('fraudTest.csv',('fraud-detection','credit-card-transactions'),'Sparkov test')
pst=profile(st,'Sparkov train','is_fraud'); psv=profile(sv,'Sparkov test','is_fraud'); display(st.head(3))
mst=missing(st,'Sparkov train',23); msv=missing(sv,'Sparkov test',23)
ids=('Unnamed: 0','cc_num','trans_num','first','last','street','zip'); times=('trans_date_trans_time','dob','unix_time')
display(attention(st,'is_fraud',ids,times))
for d,n in [(st,'train'),(sv,'test')]:
    dt=pd.to_datetime(d.trans_date_trans_time,errors='coerce'); print(n,dt.min(),'→',dt.max(),'invalid=',dt.isna().sum())
print('cc_num train/test/overlap:',st.cc_num.nunique(),sv.cc_num.nunique(),len(set(st.cc_num)&set(sv.cc_num)))
print('trans_num overlap:',len(set(st.trans_num)&set(sv.trans_num)))
display(st.groupby('is_fraud')[['amt','city_pop','unix_time']].agg(['count','mean','median','std','min','max'])); amount_plot(st,'amt','is_fraud','Sparkov')
for c in ['category','gender','state']: display(Markdown('**`'+c+'`**')); display(rate_table(st,c,'is_fraud'))

### Sparkov: capacity One-Hot + SMOTE

A = One-Hot ngây thơ mọi object, gồm `trans_num` gần unique. B = capacity sau khi đưa ID/PII và datetime raw vào danh sách review/exclude; **không thực hiện drop**. B vẫn One-Hot `merchant`, `category`, `gender`, `city`, `state`, `job`.

In [ ]:
obj=cats(st,('is_fraud',)); naive=[(st,c,False) for c in obj]
exclude={'Unnamed: 0','cc_num','trans_num','first','last','street','zip','trans_date_trans_time','dob','unix_time'}
review=[(st,c,False) for c in obj if c not in exclude]
sa=estimate([st],len(st),'is_fraud',set(),naive,'A mọi object, gồm high-card ID')
sb=estimate([st],len(st),'is_fraud',exclude,review,'B review/exclude ID + datetime raw')
display(pd.DataFrame([public(sa),public(sb)])); display(pd.DataFrame(sa['_levels'],columns=['feature','levels']).sort_values('levels',ascending=False).head(25))
ss=smote_size(st.is_fraud,sb); display(pd.DataFrame([ss]))
results['Sparkov train']={**pst,'fraud':int(st.is_fraud.sum()),'fraud_rate':float(st.is_fraud.mean()),'features_after_review':sb['total_features'],'dense_train':sb['dense_float32'],'rows_after_smote':ss['rows_after'],'dense_after_smote':ss['dense_after']}
results['Sparkov supplied test']={**psv,'fraud':int(sv.is_fraud.sum()),'fraud_rate':float(sv.is_fraud.mean()),'features_after_review':'fit on train','dense_train':'N/A','rows_after_smote':'N/A','dense_after_smote':'N/A'}
del st,sv,mst,msv; gc.collect()

### Nhận định Sparkov

- Không gộp train/test rồi random split nếu muốn giữ holdout thời gian có sẵn; encoder/scaler fit train-only.
- `trans_num` gần unique làm OHE bùng nổ. `cc_num`, tên, địa chỉ/ZIP dễ tạo memorization hơn là hành vi tổng quát.
- Datetime cần parse sau khi protocol được duyệt; chưa tự tạo hour/month/weekend/age/distance vì source chưa mô tả đầy đủ.
- SMOTE 1:1 trên hơn một triệu dòng gần nhân đôi majority và có thể là bước nặng nhất.

## 4) Tổng hợp và STOP

Nếu `EDA_NROWS != None`, mọi count/cardinality/estimate dưới đây chỉ là sample, không dùng làm kết luận nghiên cứu.

In [ ]:
display(pd.DataFrame(results).T)
display(Markdown('''
### Checklist phải khóa trước preprocessing

1. Split: MLG random/temporal; IEEE Stratified 5-Fold; Sparkov giữ supplied test và tạo validation trong train.
2. Duplicate: định nghĩa theo full row hay business key; xử lý trước split.
3. ID/time: khóa danh sách chỉ-truy-vết và cách xử lý thời gian.
4. IEEE: transaction-only hay left join identity; drop missing >50% dựa trên train fold.
5. Categorical: review numeric-coded category; không OHE ID/high-cardinality thiếu căn cứ.
6. Impute/scale/encode: fit train-only; validation/test chỉ transform.
7. SMOTE: train-only, ghi ratio và `k_neighbors=5` nếu tái lập; không cân bằng validation/test.
8. TabNet: không `.toarray()` toàn IEEE/Sparkov; kiểm thử sparse/batch và peak RAM.
9. Metrics sau này: F1 trọng tâm, kèm Precision, Recall, ROC-AUC, PR-AUC; Accuracy không làm kết luận chính.

**STOP — EDA hoàn tất. One-Hot, StandardScaler, SMOTE và model training: NOT APPLIED.**
'''))